In [1]:
#Let’s import the libraries we’ll be using.

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import defaultdict
from io import StringIO

%matplotlib inline
import seaborn as sns
sns.set_style('white')
sns.set_style('ticks')
sns.set_context('notebook')
from scipy.spatial.distance import squareform
import h5py
import allel
import zarr
import matplotlib.pyplot as plt
from cycler import cycler
import allel
import glob
import os
import pickle
import csv
#import pybedtools
from scipy.special import logit

In [2]:
# Load the Zarr callset
zarr_path = '/data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/VCFs/vcf_gz/zarrs/AllChr_newdapc_byclus_depth_ABfilt_concat.zarr'
callset = zarr.open_group(zarr_path, mode='r')
callset.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='All_chr', nodes=(Node(disabled=True…

In [3]:


# Load the data
samples_p = '/data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/VCFs/vcf_gz/Samplemd_files/Final_Dataset_bydapcclusters_remoutliers/Allchr_Whitefullfilt_Newdapc_bycluster_md.txt'
samples = pd.read_csv(samples_p, sep='\t')
print('No of Individuals:',len(samples['assgn_clus']))
# Convert 'assgn_clus' to numeric values (int or float as needed)
samples['assgn_clus'] = pd.to_numeric(samples['assgn_clus'], errors='coerce')

# Verify the data type of 'assgn_clus'
print(f"Data type of 'assgn_clus': {samples['assgn_clus'].dtype}")

# Check if 'assgn_clus' contains only the values 1, 4, 6
print(samples['assgn_clus'].value_counts())

# Define subpopulations based on numeric 'assgn_clus'
subpops = {
    'all': list(range(len(samples))),
    'MaeSot_Thailand': samples[samples['assgn_clus'] == 1].index.tolist(),
    'PusPai_Cambodia': samples[samples['assgn_clus'] == 6].index.tolist(),
    'BinhPhuoc_Long_Vietnam': samples[samples['assgn_clus'] == 4].index.tolist()
}
print(subpops['MaeSot_Thailand'][:5])

# Load variants data
variants = allel.VariantChunkedTable(callset['All_chr/variants'], 
                                     names=['POS', 'CHROM', 'numalt', 'REF', 'ALT', 'DP', 'MQ', 'QD', 'ANN_Annotation'])

# Get unique chromosome names
chrom_uniq = np.unique(variants['CHROM'])


No of Individuals: 270
Data type of 'assgn_clus': int64
assgn_clus
4    156
1     89
6     25
Name: count, dtype: int64
[0, 1, 2, 3, 4]


In [ ]:
gt_afr = gt_variant_selection.take(subpops['MaeSot_Thailand'], axis=1)
gt_afr

NameError: name 'gt_variant_selection' is not defined

In [4]:
gt_zarr = callset['All_chr/calldata/GT']
gt_zarr.info
gt = allel.HaplotypeArray(callset['All_chr/calldata/GT'][:,:,0])

genotypes = allel.HaplotypeChunkedArray(gt)
print(len(genotypes))

variants = allel.VariantChunkedTable(callset['All_chr/variants'], 
                                     names=['POS', 'CHROM', 'numalt', 'REF', 'ALT', 'DP', 'MQ', 'QD'])
gt = allel.HaplotypeArray(callset['All_chr/calldata/GT'][:,:,0])
print(gt.shape)
genotypes = allel.HaplotypeChunkedArray(gt)


480058
(480058, 270)


In [5]:
variants

<VariantChunkedTable shape=(480058,) dtype=[('POS', '<i4'), ('CHROM', 'O'), ('numalt', '<i4'), ('REF', 'O'), ('ALT', 'O', (3,)), ('DP', '<i4'), ('MQ', '<f4'), ('QD', '<f4')]
   nbytes=27.5M cbytes=2.8M cratio=9.6
   values=zarr.hierarchy.Group>

In [5]:
chrom_uniq = np.unique(variants['CHROM'])
chrom_variants = {}
chrom_pos = {}
chrom_gt = {}
for chrom in chrom_uniq:
    # Create a boolean mask for the current chromosome
    mask_chrom = variants['CHROM'] == chrom
    
    # Use compress to subset the variants for the current chromosome
    chrom_variants[chrom] = variants.compress(mask_chrom)
    
    # Extract the positions for the current chromosome from the filtered variants
    chrom_pos[chrom] = np.unique(chrom_variants[chrom]['POS'])

    # Subset genotypes for the current chromosome
    chrom_gt[chrom] = genotypes.compress(mask_chrom)
    
    
    # Calculate the range of positions for the current chromosome in base pairs
    chrom_range_bp = np.max(chrom_pos[chrom]) - np.min(chrom_pos[chrom])
    
    # Convert the range from base pairs to megabase pairs
    chrom_range_mb = chrom_range_bp / 1_000_000

    # Print the range for the current chromosome in megabase pairs
    print(f"Chromosome {chrom}: Range = {chrom_range_mb:.2f} Mb, Fulllen:{chrom_range_bp}")

chrom_range_bp

Chromosome Pf3D7_01_v3: Range = 0.48 Mb, Fulllen:482330
Chromosome Pf3D7_02_v3: Range = 0.76 Mb, Fulllen:756679
Chromosome Pf3D7_03_v3: Range = 0.93 Mb, Fulllen:932217
Chromosome Pf3D7_04_v3: Range = 1.05 Mb, Fulllen:1052478
Chromosome Pf3D7_05_v3: Range = 1.28 Mb, Fulllen:1283140
Chromosome Pf3D7_06_v3: Range = 1.22 Mb, Fulllen:1221885
Chromosome Pf3D7_07_v3: Range = 1.30 Mb, Fulllen:1304387
Chromosome Pf3D7_08_v3: Range = 1.29 Mb, Fulllen:1291977
Chromosome Pf3D7_09_v3: Range = 1.39 Mb, Fulllen:1393634
Chromosome Pf3D7_10_v3: Range = 1.50 Mb, Fulllen:1502764
Chromosome Pf3D7_11_v3: Range = 1.89 Mb, Fulllen:1893297
Chromosome Pf3D7_12_v3: Range = 2.10 Mb, Fulllen:2103023
Chromosome Pf3D7_13_v3: Range = 2.72 Mb, Fulllen:2717429
Chromosome Pf3D7_14_v3: Range = 3.22 Mb, Fulllen:3219912


3219912

##### Subsetting Individuals from a population by genotype

In [39]:
gt_data_by_chrom = {}
all_maesot = []

##### Subsetting Individuals from a population by allele counts

In [25]:
ac = genotypes.count_alleles()
ac

<AlleleCountsChunkedArray shape=(480058, 2) dtype=int32 chunks=(60008, 2)
   nbytes=3.7M cbytes=297.2K cratio=12.6
   compression=blosc compression_opts={'cname': 'lz4', 'clevel': 5, 'shuffle': 1, 'blocksize': 0}
   values=zarr.core.Array>

In [6]:

for chrom in chrom_uniq:
    # Create a boolean mask for the current chromosome
    mask_chrom = variants['CHROM'] == chrom
    
    # Use compress to subset the variants for the current chromosome
    chrom_variants[chrom] = variants.compress(mask_chrom)
    
    # Extract the positions for the current chromosome from the filtered variants
    chrom_pos[chrom] = np.unique(chrom_variants[chrom]['POS'])

    # Subset genotypes for the current chromosome
    chrom_gt[chrom] = genotypes.compress(mask_chrom)

    # Define window and step sizes
    window_size = 5000
    step_size = 2500

    

# Function to get positions and genotypes for a chromosome
def get_chrom_data(chrom):
    pos = allel.SortedIndex(chrom_pos[chrom])
    gt = chrom_gt[chrom]
    genotypes = allel.HaplotypeChunkedArray(gt)
    return pos, genotypes

# Function to get annotations for a chromosome
def get_annotations(chrom):
    return chrom_variants[chrom]['ANN_Annotation']

# Function to get allele counts for subpopulations
def get_ac_subpops(chrom, genotypes):
    return genotypes.count_alleles_subpops(subpops=subpops, max_allele=2)

# Function to get masks for different variant types
def get_masks(an):
    # Define variant categories broadly considered as neutrally evolving
    synonymous_variants = [
        'synonymous_variant',
        'splice_region_variant&synonymous_variant',
        #'stop_retained_variant',
        '5_prime_UTR_variant',
        '3_prime_UTR_variant',
        'intron_variant',
        #'intergenic_region',
        'non_coding_transcript_variant',
        #'intragenic_variant',
       # '5_prime_UTR_premature_start_codon_gain_variant'
    ]
    
    missense_mask = np.isin(an, ['missense_variant', 'missense_variant&splice_region_variant'])
    syn_mask = np.isin(an, synonymous_variants)
    intr_mask = np.isin(an, ['intergenic_region'])  # This can be kept as is or modified as needed
    
    return missense_mask, syn_mask, intr_mask


# Function to calculate statistics for a variant type and subpopulation
def calculate_stats(variants, ac_subpops):
    nd_win_1, windows, _, _ = allel.windowed_diversity(
        variants, ac_subpops['MaeSot_Thailand'], step=step_size, size=window_size)
    nd_win_2, _, _, _ = allel.windowed_diversity(
        variants, ac_subpops['PurPai_Cambodia'], step=step_size, size=window_size)
    nd_win_3, _, _, _ = allel.windowed_diversity(
        variants, ac_subpops['BinhPhuoc_Long_Vietnam'], step=step_size, size=window_size)

    S1, _, _ = allel.windowed_statistic(variants, ac_subpops['MaeSot_Thailand'].is_segregating()[:],
                                        statistic=np.count_nonzero, step=step_size, size=window_size, fill=0)
    S2, _, _ = allel.windowed_statistic(variants, ac_subpops['PurPai_Cambodia'].is_segregating()[:],
                                        statistic=np.count_nonzero, step=step_size, size=window_size, fill=0)
    S3, _, _ = allel.windowed_statistic(variants, ac_subpops['BinhPhuoc_Long_Vietnam'].is_segregating()[:],
                                        statistic=np.count_nonzero, step=step_size, size=window_size, fill=0)

    wt_1, _, _, _ = allel.windowed_watterson_theta(
        variants, ac_subpops['MaeSot_Thailand'], step=step_size, size=window_size)
    wt_2, _, _, _ = allel.windowed_watterson_theta(
        variants, ac_subpops['PurPai_Cambodia'], step=step_size, size=window_size)
    wt_3, _, _, _ = allel.windowed_watterson_theta(
        variants, ac_subpops['BinhPhuoc_Long_Vietnam'], step=step_size, size=window_size)

    d1, _, _ = allel.windowed_tajima_d(
        variants, ac_subpops['MaeSot_Thailand'], step=step_size, size=window_size)
    d2, _, _ = allel.windowed_tajima_d(
        variants, ac_subpops['PurPai_Cambodia'], step=step_size, size=window_size)
    d3, _, _ = allel.windowed_tajima_d(
        variants, ac_subpops['BinhPhuoc_Long_Vietnam'], step=step_size, size=window_size)

    return windows, (nd_win_1, nd_win_2, nd_win_3), (S1, S2, S3), (wt_1, wt_2, wt_3), (d1, d2, d3)

# Function to convert genomic positions to kilobases and calculate midpoints
def get_midpoints(windows):
    midpoint_positions = (windows[:, 0] + windows[:, 1]) / 2
    return midpoint_positions / 1000  # Assuming positions in base pairs


In [7]:
window_size = 5000
step_size = 2500

# Final data storage
sample_data_by_chrom = {}

# Tracking stats
total_vars_missing_cleaned = 0
total_vars = 0

# Loop through each chromosome
for chrom in chrom_uniq:
    print(f"\nProcessing chromosome: {chrom}")
    
    # Step 1: Subset variants/genotypes by chromosome
    mask_chrom = variants['CHROM'] == chrom
    chrom_variants[chrom] = variants.compress(mask_chrom)
    chrom_pos[chrom] = chrom_variants[chrom]['POS'][:]
    chrom_gt[chrom] = genotypes.compress(mask_chrom)
    
    
    # Step 2: Remove sites with any missing genotype
    is_missing = chrom_gt[chrom].is_missing()[:]
    valid_sites_mask = ~np.any(is_missing, axis=1)
    
    chrom_gt[chrom] = chrom_gt[chrom].compress(valid_sites_mask)
    chrom_variants[chrom] = chrom_variants[chrom].compress(valid_sites_mask)
    chrom_pos[chrom] = chrom_pos[chrom][valid_sites_mask]

    num_cleaned = np.sum(valid_sites_mask)
    total_vars_missing_cleaned += num_cleaned
    total_vars += np.sum(mask_chrom)
    print(f"Removed {np.sum(mask_chrom) - num_cleaned} missing sites; {num_cleaned} remain.")

    # Step 3: Count alleles for each subpopulation
    ac_subpops = chrom_gt[chrom].count_alleles_subpops(subpops, max_allele=2)

    # Output structure
    gt_valid_by_subpop = {}
    pos_by_subpop = {}

    # Subpopulation processing
    target_subpops = [
        ('MaeSot_Thailand', 'GT_MaeSot', 'POS_MaeSot'),
        ('BinhPhuoc_Long_Vietnam', 'GT_Binh', 'POS_Binh')
    ]

    for subpop_key, gt_key, pos_key in target_subpops:
        # Step 4: Identify segregating sites for subpop
        is_seg = ac_subpops[subpop_key].is_segregating()[:]

        # Step 5: Extract genotypes and positions
        gt = chrom_gt[chrom].take(subpops[subpop_key], axis=1)
        gt_seg = gt.compress(is_seg, axis=0)
        gt_valid = pd.DataFrame(gt_seg).T
        pos_seg = chrom_pos[chrom][is_seg]

        gt_valid_by_subpop[gt_key] = gt_valid
        pos_by_subpop[pos_key] = pos_seg

        print(f"{gt_key}: {gt_valid.shape[0]} samples, {gt_valid.shape[1]} segregating sites")

    # Step 6: Store results
    sample_data_by_chrom[chrom] = {
        'CHROM': chrom,
        'GT_MaeSot': gt_valid_by_subpop.get('GT_MaeSot', pd.DataFrame()),
        'POS_MaeSot': pos_by_subpop.get('POS_MaeSot', np.array([])),
        'GT_Binh': gt_valid_by_subpop.get('GT_Binh', pd.DataFrame()),
        'POS_Binh': pos_by_subpop.get('POS_Binh', np.array([]))
    }

print(f"\nTotal variants processed: {total_vars}")
print(f"Total variants remaining after missing site removal: {total_vars_missing_cleaned}")





Processing chromosome: Pf3D7_01_v3
Removed 1453 missing sites; 8290 remain.
GT_MaeSot: 89 samples, 319 segregating sites
GT_Binh: 156 samples, 461 segregating sites

Processing chromosome: Pf3D7_02_v3
Removed 2751 missing sites; 13444 remain.
GT_MaeSot: 89 samples, 367 segregating sites
GT_Binh: 156 samples, 606 segregating sites

Processing chromosome: Pf3D7_03_v3
Removed 2453 missing sites; 17616 remain.
GT_MaeSot: 89 samples, 520 segregating sites
GT_Binh: 156 samples, 825 segregating sites

Processing chromosome: Pf3D7_04_v3
Removed 3398 missing sites; 17939 remain.
GT_MaeSot: 89 samples, 685 segregating sites
GT_Binh: 156 samples, 1121 segregating sites

Processing chromosome: Pf3D7_05_v3
Removed 4147 missing sites; 23818 remain.
GT_MaeSot: 89 samples, 555 segregating sites
GT_Binh: 156 samples, 1055 segregating sites

Processing chromosome: Pf3D7_06_v3
Removed 3182 missing sites; 23019 remain.
GT_MaeSot: 89 samples, 515 segregating sites
GT_Binh: 156 samples, 899 segregating sit

 **Sites file**

- with the first line detailing the number of sequences/genotypes, 
- the number of sites in the alignment and 
- a flag (1 or 2) that details whether the data is haplotype/phased (1) or genotype/unphased (2).

**Locs file**

- The locs file has on the first line the number of SNPs (segregating sites),
- the total length of the region analysed, 
- and a flag (L or C) that details whether a model of crossing-over or gene conversion is fitted 

In [1]:
output_dir= '/data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/ld_hat'

In [137]:
for chrom, data in sample_data_by_chrom.items():
    # Iterate over subpopulations
    for subpop in ['GT_MaeSot', 'GT_Binh']:
        
        # Output file name
        filename = f"{output_dir}/{subpop}_{chrom}_sample_gt_counts.txt"

        # Determine number of sequences and sites
        num_sequences = data[subpop].shape[0]
        num_sites = data[subpop].shape[1]

        # Set flag: 2 = unphased genotypes (standard VCF style)
        phased_flag = 1

        # First line format: "<num_sequences> <num_sites> <flag>"
        line0 = f"{num_sequences} {num_sites} {phased_flag}"

        # Function to split lines if they exceed 2000 characters
        def split_line(line):
            return [line[i:i+2000] for i in range(0, len(line), 2000)]

        # Write header line
        with open(filename, "w") as f:
            for chunk in split_line(line0):
                f.write(chunk + "\n")

        # Write genotype data
        with open(filename, "a") as f:
            for index, row in data[subpop].iterrows():
                line1 = f'>Genotype{index}'
                line2 = ''.join(map(str, row.values))
                
                f.write(line1 + "\n")
                for chunk in split_line(line2):
                    f.write(chunk + "\n")

        print(f"Saved: {filename}")



Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/GT_MaeSot_Pf3D7_01_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/GT_Binh_Pf3D7_01_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/GT_MaeSot_Pf3D7_02_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/GT_Binh_Pf3D7_02_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/GT_MaeSot_Pf3D7_03_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/GT_Binh_Pf3D7_03_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/GT_MaeSot_Pf3D7_04_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_h

In [139]:
for chrom, data in sample_data_by_chrom.items():
    # Iterate over subpopulations
    for subpop in ['GT_MaeSot', 'GT_Binh']:
        
        # Output file name
        filename = f"{output_dir}/{subpop}_{chrom}_sample_gt_counts.txt"

        # Determine number of sequences and sites
        num_sequences = data[subpop].shape[0]
        num_sites = data[subpop].shape[1]

        # Set flag: 2 = unphased genotypes (standard VCF style)
        phased_flag = 1

        # First line format: "<num_sequences> <num_sites> <flag>"
        line0 = f"{num_sequences} {num_sites} {phased_flag}"

        # Write header line
        with open(filename, "w") as f:
            f.write(line0 + "\n")

        # Write genotype data
        for index, row in data[subpop].iterrows():
            line1 = f'>Genotype{index}'
            line2 = ''.join(map(str, row.values))
                    
            with open(filename, "a") as f:
                f.write(line1 + "\n")
                f.write(line2 + "\n")

        print(f"Saved: {filename}")

Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/ld_hat/GT_MaeSot_Pf3D7_01_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/ld_hat/GT_Binh_Pf3D7_01_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/ld_hat/GT_MaeSot_Pf3D7_02_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/ld_hat/GT_Binh_Pf3D7_02_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/ld_hat/GT_MaeSot_Pf3D7_03_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/ld_hat/GT_Binh_Pf3D7_03_v3_sample_gt_counts.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/ld_hat/GT_MaeSot_Pf3D7_04_v3_sample_gt_counts.txt
Saved: /data/proj2/home/st

In [135]:

for chrom, data in sample_data_by_chrom.items():
    filename = f"{output_dir}/{chrom}_seg_sites.txt"

    # Calculate the number of SNPs
    num_snps = len(data['POS'])

    # Calculate region length (bp)
    chrom_range_bp = np.max(data['POS']) / 1000

    # Flag for crossing-over model
    model_flag = "L"

    # Write the header line with num_snps, chrom_range_bp, and model_flag
    with open(filename, "w") as f:
        f.write(f"{num_snps} {chrom_range_bp} {model_flag}\n")

    # Append the positions (segregating sites) converted to kb
    for pos in data['POS']:
        pos_kb = pos / 1000
        with open(filename, "a") as f:
            f.write(f"{pos_kb:.3f}\n")  # format to 3 decimals

    print(f"Saved: {filename}")



Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Pf3D7_01_v3_seg_sites.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Pf3D7_02_v3_seg_sites.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Pf3D7_03_v3_seg_sites.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Pf3D7_04_v3_seg_sites.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Pf3D7_05_v3_seg_sites.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Pf3D7_06_v3_seg_sites.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Pf3D7_07_v3_seg_sites.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Pf3D7_08_v3_seg_sites.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/L

### Setting missing sites to ?


In [ ]:
for chrom in chrom_uniq:
    print(f"\nProcessing chromosome: {chrom}")
    
    # Mask and subset variants/genotypes
    mask_chrom = variants['CHROM'] == chrom
    chrom_variants[chrom] = variants.compress(mask_chrom)
    chrom_pos[chrom] = chrom_variants[chrom]['POS'][:]
    chrom_gt[chrom] = genotypes.compress(mask_chrom)

    total_vars_missing_cleaned += len(chrom_pos[chrom])
    total_vars += np.sum(mask_chrom)

    # Count alleles per subpopulation
    ac_subpops = chrom_gt[chrom].count_alleles_subpops(subpops, max_allele=2)

    # Initialize dictionaries to store per-subpop data
    gt_valid_by_subpop = {}
    pos_by_subpop = {}

    # Subpopulation keys and their output labels
    target_subpops = [
        ('MaeSot_Thailand', 'GT_MaeSot', 'POS_MaeSot'),
        ('BinhPhuoc_Long_Vietnam', 'GT_Binh', 'POS_Binh')
    ]

    for subpop_key, gt_key, pos_key in target_subpops:
        # Segregating site mask for this subpop
        is_seg = ac_subpops[subpop_key].is_segregating()[:]

        # Genotypes for this subpop
        gt = chrom_gt[chrom].take(subpops[subpop_key], axis=1)

        # Genotypes filtered to segregating sites
        gt_seg = gt.compress(is_seg, axis=0)

        # Convert to pandas DataFrame and transpose to samples × sites
        gt_valid = pd.DataFrame(gt_seg).T

        # Get segregating variant positions
        pos_seg = chrom_pos[chrom][is_seg]

        # Store results
        gt_valid_by_subpop[gt_key] = gt_valid
        pos_by_subpop[pos_key] = pos_seg

        print(f"{gt_key}: {gt_valid.shape[0]} samples, {gt_valid.shape[1]} segregating sites")

    # Store result for this chromosome
    variant_info = {
        'CHROM': np.array([chrom]),
        'GT_MaeSot': gt_valid_by_subpop.get('GT_MaeSot', pd.DataFrame()),
        'POS_MaeSot': pos_by_subpop.get('POS_MaeSot', np.array([])),
        'GT_Binh': gt_valid_by_subpop.get('GT_Binh', pd.DataFrame()),
        'POS_Binh': pos_by_subpop.get('POS_Binh', np.array([]))
    }

    sample_data_by_chrom[chrom] = variant_info




Processing chromosome: Pf3D7_01_v3
GT_MaeSot: 89 samples, 522 segregating sites
GT_Binh: 156 samples, 707 segregating sites

Processing chromosome: Pf3D7_02_v3
GT_MaeSot: 89 samples, 694 segregating sites
GT_Binh: 156 samples, 997 segregating sites

Processing chromosome: Pf3D7_03_v3
GT_MaeSot: 89 samples, 755 segregating sites
GT_Binh: 156 samples, 1135 segregating sites

Processing chromosome: Pf3D7_04_v3
GT_MaeSot: 89 samples, 1186 segregating sites
GT_Binh: 156 samples, 1726 segregating sites

Processing chromosome: Pf3D7_05_v3
GT_MaeSot: 89 samples, 932 segregating sites
GT_Binh: 156 samples, 1576 segregating sites

Processing chromosome: Pf3D7_06_v3
GT_MaeSot: 89 samples, 775 segregating sites
GT_Binh: 156 samples, 1244 segregating sites

Processing chromosome: Pf3D7_07_v3
GT_MaeSot: 89 samples, 1137 segregating sites
GT_Binh: 156 samples, 1758 segregating sites

Processing chromosome: Pf3D7_08_v3
GT_MaeSot: 89 samples, 971 segregating sites
GT_Binh: 156 samples, 1586 segregatin

In [33]:
sample_data_by_chrom['Pf3D7_01_v3']

{'CHROM': array(['Pf3D7_01_v3'], dtype='<U11'),
 'POS': array([ 92914,  92918,  92935, ..., 571807, 571809, 575244], dtype=int32),
 'GT_MaeSot':     0    1    2    3    4    5    6    7    8    9    ...  512  513  514  515  \
 0     0    0    0   -1    0    0   -1    0    0    0  ...    0   -1   -1   -1   
 1     0    0    0    0    0    0    1    0    0    0  ...    0    0    0    0   
 2     0    0    0    0    0    0    1    0    1    0  ...    0    0    0    0   
 3     0    0    0    0    1    0    0    0    0    0  ...    1    1    0    0   
 4     0    0    0    0    0    0    1    0    0    0  ...    0    1    0    0   
 ..  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
 84    0    0    1    0    1    0    0    0    0    0  ...    0    0    0    0   
 85    0    0    0    0    0    0    1    0    0    0  ...    0    0    0    0   
 86    0    0    0    0    0    0    1    0    0   -1  ...    0    0    0    0   
 87    0    0    0    0    0    0   

In [46]:
output_dir= '/data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Samples_50'

In [9]:
# === Write output with '?' for missing data ===

for chrom, data in sample_data_by_chrom.items():
    for subpop in ['GT_MaeSot', 'GT_Binh']:
        filename = f"{output_dir}/{subpop}_{chrom}_sample_gt_counts.txt"
        num_sequences = data[subpop].shape[0]
        num_sites = data[subpop].shape[1]
        phased_flag = 1  # Indicates phased genotypes
        line0 = f"{num_sequences} {num_sites} {phased_flag}"

        def split_line(line):
            return [line[i:i+2000] for i in range(0, len(line), 2000)]

        # Write header line (split if too long)
        with open(filename, "w") as f:
            for chunk in split_line(line0):
                f.write(chunk + "\n")

        # Write genotype data, replacing -1 (missing) with '?'
        with open(filename, "a") as f:
            for index, row in data[subpop].iterrows():
                line1 = f'>Genotype{index}'

                def replace_missing(x):
                    try:
                        val = int(x)
                        return '?' if val == -1 else str(val)
                    except Exception:
                        return '?'

                line2 = ''.join(replace_missing(x) for x in row.values)

                f.write(line1 + "\n")
                for chunk in split_line(line2):
                    f.write(chunk + "\n")



In [10]:
for chrom, data in sample_data_by_chrom.items():
    for subpop_key in ['MaeSot', 'Binh']:
        pos_key = f"POS_{subpop_key}"
        pos_array = data.get(pos_key, np.array([]))

        if len(pos_array) == 0:
            print(f"Skipping {chrom} - {subpop_key}: no segregating sites.")
            continue

        filename = f"{output_dir}/{chrom}_seg_sites_{subpop_key}.txt"

        # Number of SNPs and chromosomal length in kb
        num_snps = len(pos_array)
        chrom_range_bp = np.max(pos_array) / 1000  # in kb
        model_flag = "L"

        with open(filename, "w") as f:
            f.write(f"{num_snps} {chrom_range_bp} {model_flag}\n")
            for pos in pos_array:
                pos_kb = pos / 1000
                f.write(f"{pos_kb:.3f}\n")

        print(f"Saved: {filename}")


Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/wo_missingness/Pf3D7_01_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/wo_missingness/Pf3D7_01_v3_seg_sites_Binh.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/wo_missingness/Pf3D7_02_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/wo_missingness/Pf3D7_02_v3_seg_sites_Binh.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/wo_missingness/Pf3D7_03_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/wo_missingness/Pf3D7_03_v3_seg_sites_Binh.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/wo_missingness/Pf3D7_04_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srini

#### WIth removal of missing inds

In [13]:
# Compute missing data per sample
is_missing = genotypes.is_missing()[:]
sample_missing_counts = np.sum(is_missing, axis=0)

# Get indices of 50 samples with least missing data
least_missing_indices = np.argsort(sample_missing_counts)[:50]

# Optional: get sample names if needed
#top_50_sample_names = [genotypes.samples[i] for i in least_missing_indices]

# Filter genotypes to only those samples
genotypes_nomiss = genotypes.take(least_missing_indices, axis=1)

In [17]:
len(sample_missing_counts)

270

In [ ]:
window_size = 5000
step_size = 2500

# Final data storage
sample_data_by_chrom = {}

# Tracking stats
total_vars_missing_cleaned = 0
total_vars = 0

# Loop through each chromosome
for chrom in chrom_uniq:
    print(f"\nProcessing chromosome: {chrom}")
    
    # Step 1: Subset variants/genotypes by chromosome
    mask_chrom = variants['CHROM'] == chrom
    chrom_variants[chrom] = variants.compress(mask_chrom)
    chrom_pos[chrom] = chrom_variants[chrom]['POS'][:]
    chrom_gt[chrom] = genotypes.compress(mask_chrom)
    
    
    # Step 2: Remove sites with any missing genotype
    is_missing = chrom_gt[chrom].is_missing()[:]
    valid_sites_mask = ~np.any(is_missing, axis=1)
    
    chrom_gt[chrom] = chrom_gt[chrom].compress(valid_sites_mask)
    chrom_variants[chrom] = chrom_variants[chrom].compress(valid_sites_mask)
    chrom_pos[chrom] = chrom_pos[chrom][valid_sites_mask]

    num_cleaned = np.sum(valid_sites_mask)
    total_vars_missing_cleaned += num_cleaned
    total_vars += np.sum(mask_chrom)
    print(f"Removed {np.sum(mask_chrom) - num_cleaned} missing sites; {num_cleaned} remain.")

    # Step 3: Count alleles for each subpopulation
    ac_subpops = chrom_gt[chrom].count_alleles_subpops(subpops, max_allele=2)

    # Output structure
    gt_valid_by_subpop = {}
    pos_by_subpop = {}

    # Subpopulation processing
    target_subpops = [
        ('MaeSot_Thailand', 'GT_MaeSot', 'POS_MaeSot'),
        ('BinhPhuoc_Long_Vietnam', 'GT_Binh', 'POS_Binh')
    ]

    for subpop_key, gt_key, pos_key in target_subpops:
        # Step 4: Identify segregating sites for subpop
        is_seg = ac_subpops[subpop_key].is_segregating()[:]

        # Step 5: Extract genotypes and positions
        gt = chrom_gt[chrom].take(subpops[subpop_key], axis=1)
        gt_seg = gt.compress(is_seg, axis=0)
        gt_valid = pd.DataFrame(gt_seg).T
        pos_seg = chrom_pos[chrom][is_seg]

        gt_valid_by_subpop[gt_key] = gt_valid
        pos_by_subpop[pos_key] = pos_seg

        print(f"{gt_key}: {gt_valid.shape[0]} samples, {gt_valid.shape[1]} segregating sites")

    # Step 6: Store results
    sample_data_by_chrom[chrom] = {
        'CHROM': chrom,
        'GT_MaeSot': gt_valid_by_subpop.get('GT_MaeSot', pd.DataFrame()),
        'POS_MaeSot': pos_by_subpop.get('POS_MaeSot', np.array([])),
        'GT_Binh': gt_valid_by_subpop.get('GT_Binh', pd.DataFrame()),
        'POS_Binh': pos_by_subpop.get('POS_Binh', np.array([]))
    }

print(f"\nTotal variants processed: {total_vars}")
print(f"Total variants remaining after missing site removal: {total_vars_missing_cleaned}")



In [ ]:
gt_subpop = {}

# Step 4: For each subpop
for subpop_key, gt_key, pos_key in target_subpops:
        subpop_indices = subpops[subpop_key]
        gt_sub = genotypes.take(subpop_indices, axis=1)
        is_missing_sub = gt_sub.is_missing()[:]

        # Step 5: Select top 50 samples with least missing genotypes
        missing_per_sample = np.sum(is_missing_sub, axis=0)
        top50_indices = np.argsort(missing_per_sample)[:50]
        selected_indices = [subpop_indices[i] for i in top50_indices]
        
        # Extract genotypes for top 50
        gt_top50 = genotypes.take(selected_indices, axis=1)
        
        gt_subpop[gt_key] = gt_top50
        
        print(f"{gt_key}: {gt_top50.shape[0]} sites, {gt_top50.shape[1]}  samples")

GT_MaeSot: 480058 sites, 50  samples
GT_Binh: 480058 sites, 50  samples


In [42]:
window_size = 5000
step_size = 2500

# Final data storage
sample_data_by_chrom_ms = {}

# Subpopulation of interest
subpop_key = 'MaeSot_Thailand'
gt_key = 'GT_MaeSot'
pos_key = 'POS_MaeSot'

# Loop through each chromosome
for chrom in chrom_uniq:
    print(f"\nProcessing chromosome: {chrom}")
    
    # Step 1: Subset variants/genotypes by chromosome
    mask_chrom = variants['CHROM'] == chrom
    chrom_variants[chrom] = variants.compress(mask_chrom)
    chrom_pos[chrom] = chrom_variants[chrom]['POS'][:]
    gt_top50 = gt_subpop['GT_MaeSot'].compress(mask_chrom)

    # Step 5: Identify segregating sites among top 50 samples
    ac_top50 = gt_top50.count_alleles(max_allele=2)
    is_seg = ac_top50.is_segregating()[:]

    gt_seg = gt_top50.compress(is_seg, axis=0)
    gt_valid = pd.DataFrame(gt_seg).T
    pos_seg = chrom_pos[chrom][is_seg]
    

    # Step 6: Store results
    sample_data_by_chrom_ms[chrom] = {
        'CHROM': chrom,
        'GT_MaeSot': gt_valid,
        'POS_MaeSot': pos_seg,
        }

    print(f"{gt_key}: {gt_valid.shape[0]} samples, {gt_valid.shape[1]} segregating sites")




Processing chromosome: Pf3D7_01_v3
GT_MaeSot: 50 samples, 455 segregating sites

Processing chromosome: Pf3D7_02_v3
GT_MaeSot: 50 samples, 586 segregating sites

Processing chromosome: Pf3D7_03_v3
GT_MaeSot: 50 samples, 633 segregating sites

Processing chromosome: Pf3D7_04_v3
GT_MaeSot: 50 samples, 1060 segregating sites

Processing chromosome: Pf3D7_05_v3
GT_MaeSot: 50 samples, 785 segregating sites

Processing chromosome: Pf3D7_06_v3
GT_MaeSot: 50 samples, 668 segregating sites

Processing chromosome: Pf3D7_07_v3
GT_MaeSot: 50 samples, 983 segregating sites

Processing chromosome: Pf3D7_08_v3
GT_MaeSot: 50 samples, 843 segregating sites

Processing chromosome: Pf3D7_09_v3
GT_MaeSot: 50 samples, 906 segregating sites

Processing chromosome: Pf3D7_10_v3
GT_MaeSot: 50 samples, 1142 segregating sites

Processing chromosome: Pf3D7_11_v3
GT_MaeSot: 50 samples, 1208 segregating sites

Processing chromosome: Pf3D7_12_v3
GT_MaeSot: 50 samples, 1131 segregating sites

Processing chromosome: 

In [38]:
# Final data storage
sample_data_by_chrom_bv = {}

# Subpopulation of interest
subpop_key = 'BinhPhuoc_Long_Vietnam'
gt_key = 'GT_Binh'
pos_key = 'POS_Binh'

# Loop through each chromosome
for chrom in chrom_uniq:
    print(f"\nProcessing chromosome: {chrom}")
    
    # Step 1: Subset variants/genotypes by chromosome
    mask_chrom = variants['CHROM'] == chrom
    chrom_variants[chrom] = variants.compress(mask_chrom)
    chrom_pos[chrom] = chrom_variants[chrom]['POS'][:]
    gt_top50 = gt_subpop[gt_key].compress(mask_chrom)

    # Step 5: Identify segregating sites among top 50 samples
    ac_top50 = gt_top50.count_alleles(max_allele=2)
    is_seg = ac_top50.is_segregating()[:]

    gt_seg = gt_top50.compress(is_seg, axis=0)
    gt_valid = pd.DataFrame(gt_seg).T
    pos_seg = chrom_pos[chrom][is_seg]
    

    # Step 6: Store results
    sample_data_by_chrom_bv[chrom] = {
         'CHROM': chrom,
         'GT_Binh': gt_valid,
        'POS_Binh': pos_seg,
        }

    print(f"{gt_key}: {gt_valid.shape[0]} samples, {gt_valid.shape[1]} segregating sites")



Processing chromosome: Pf3D7_01_v3
GT_Binh: 50 samples, 537 segregating sites

Processing chromosome: Pf3D7_02_v3
GT_Binh: 50 samples, 736 segregating sites

Processing chromosome: Pf3D7_03_v3
GT_Binh: 50 samples, 802 segregating sites

Processing chromosome: Pf3D7_04_v3
GT_Binh: 50 samples, 1335 segregating sites

Processing chromosome: Pf3D7_05_v3
GT_Binh: 50 samples, 1135 segregating sites

Processing chromosome: Pf3D7_06_v3
GT_Binh: 50 samples, 927 segregating sites

Processing chromosome: Pf3D7_07_v3
GT_Binh: 50 samples, 1329 segregating sites

Processing chromosome: Pf3D7_08_v3
GT_Binh: 50 samples, 1201 segregating sites

Processing chromosome: Pf3D7_09_v3
GT_Binh: 50 samples, 1112 segregating sites

Processing chromosome: Pf3D7_10_v3
GT_Binh: 50 samples, 1399 segregating sites

Processing chromosome: Pf3D7_11_v3
GT_Binh: 50 samples, 1538 segregating sites

Processing chromosome: Pf3D7_12_v3
GT_Binh: 50 samples, 1623 segregating sites

Processing chromosome: Pf3D7_13_v3
GT_Binh:

In [52]:
# === Write output with '?' for missing data ===
subpop = 'GT_MaeSot'
for chrom, data in sample_data_by_chrom_ms.items():
    
    filename = f"{output_dir}/{subpop}_{chrom}_sample_gt_counts.txt"
    num_sequences = data[subpop].shape[0]
    num_sites = data[subpop].shape[1]
    phased_flag = 1  # Indicates phased genotypes
    line0 = f"{num_sequences} {num_sites} {phased_flag}"

    def split_line(line):
            return [line[i:i+2000] for i in range(0, len(line), 2000)]

    # Write header line (split if too long)
    with open(filename, "w") as f:
            for chunk in split_line(line0):
                f.write(chunk + "\n")

    # Write genotype data, replacing -1 (missing) with '?'
    with open(filename, "a") as f:
        for index, row in data[subpop].iterrows():
            print(row)
            line1 = f'>Genotype{index}'

            def replace_missing(x):
                try:
                    val = int(x)
                    return '?' if val == -1 else str(val)
                except Exception:
                    return '?'

            line2 = ''.join(replace_missing(x) for x in row.values)

            f.write(line1 + "\n")
            for chunk in split_line(line2):
                f.write(chunk + "\n")


0      0
1      0
2      0
3      0
4      0
      ..
450    0
451    0
452    1
453    0
454    1
Name: 0, Length: 455, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
450    1
451    0
452    1
453    0
454    0
Name: 1, Length: 455, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
450    1
451    0
452    1
453    0
454    0
Name: 2, Length: 455, dtype: int8
0      0
1      0
2      0
3      0
4      1
      ..
450    0
451    0
452    0
453    0
454    0
Name: 3, Length: 455, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
450    0
451    0
452    0
453    0
454    0
Name: 4, Length: 455, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
450    1
451    0
452    1
453    0
454    0
Name: 5, Length: 455, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
450    1
451    0
452    1
453    0
454    0
Name: 6, Length: 455, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
450    0
451   

In [53]:
# === Write output with '?' for missing data ===
subpop = 'GT_Binh'
for chrom, data in sample_data_by_chrom_bv.items():
    
    filename = f"{output_dir}/{subpop}_{chrom}_sample_gt_counts.txt"
    num_sequences = data[subpop].shape[0]
    num_sites = data[subpop].shape[1]
    phased_flag = 1  # Indicates phased genotypes
    line0 = f"{num_sequences} {num_sites} {phased_flag}"

    def split_line(line):
            return [line[i:i+2000] for i in range(0, len(line), 2000)]

    # Write header line (split if too long)
    with open(filename, "w") as f:
            for chunk in split_line(line0):
                f.write(chunk + "\n")

    # Write genotype data, replacing -1 (missing) with '?'
    with open(filename, "a") as f:
        for index, row in data[subpop].iterrows():
            print(row)
            line1 = f'>Genotype{index}'

            def replace_missing(x):
                try:
                    val = int(x)
                    return '?' if val == -1 else str(val)
                except Exception:
                    return '?'

            line2 = ''.join(replace_missing(x) for x in row.values)

            f.write(line1 + "\n")
            for chunk in split_line(line2):
                f.write(chunk + "\n")


0      0
1      1
2      0
3      0
4      0
      ..
532    0
533    0
534    0
535    0
536    0
Name: 0, Length: 537, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
532    0
533    0
534    1
535    1
536    0
Name: 1, Length: 537, dtype: int8
0      0
1      0
2      1
3      0
4      0
      ..
532    0
533    0
534    0
535    0
536    0
Name: 2, Length: 537, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
532    0
533    0
534    0
535    0
536    0
Name: 3, Length: 537, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
532    0
533    0
534    0
535    0
536    0
Name: 4, Length: 537, dtype: int8
0      0
1      0
2      0
3     -1
4      0
      ..
532    0
533    0
534    0
535    0
536    0
Name: 5, Length: 537, dtype: int8
0      0
1      0
2      0
3      0
4      1
      ..
532    0
533    0
534    0
535    0
536    0
Name: 6, Length: 537, dtype: int8
0      0
1      0
2      0
3      0
4      0
      ..
532    0
533   

In [59]:
sample_data_by_chrom_bv

{'Pf3D7_01_v3': {'CHROM': 'Pf3D7_01_v3',
  'GT_Binh':     0    1    2    3    4    5    6    7    8    9    ...  527  528  529  530  \
  0     0    1    0    0    0    1    1    0    0    0  ...    0    0    0    0   
  1     0    0    0    0    0    0    1    0    0    0  ...    0    0    0    0   
  2     0    0    1    0    0    0    1    0    0    0  ...    1    0    0    0   
  3     0    0    0    0    0    0    1    0    0    0  ...    0    0    0    0   
  4     0    0    0    0    0    0    0    0    0    0  ...    0    0    0    0   
  5     0    0    0   -1    0    0   -1    0    0    0  ...    0    0    0    0   
  6     0    0    0    0    1    0    1    0    0    1  ...    0    0    0    1   
  7     0    0    0    0    0    0    1    0    0    0  ...    0    0    0    0   
  8     0    0    0    0    0    0    1    0    0    0  ...    0    0    0    0   
  9     0    0    0    0    0    0    1    0    0    0  ...    0    0    0    0   
  10    0    0    0    0    0    0 

In [ ]:
#subpop_key='Binh'
subpop_key='MaeSot'
sample_data_by_chrom = sample_data_by_chrom_ms
#sample_data_by_chrom = sample_data_by_chrom_bv

In [ ]:
for chrom, data in sample_data_by_chrom.items():
        #print(data)
        pos_key = f"POS_{subpop_key}"
        pos_array = data.get(pos_key, np.array([]))
        #print(pos_array)
        if len(pos_array) == 0:
            print(f"Skipping {chrom} - {subpop_key}: no segregating sites.")
            continue

        filename = f"{output_dir}/{chrom}_seg_sites_{subpop_key}.txt"

        # Number of SNPs and chromosomal length in kb
        num_snps = len(pos_array)
        chrom_range_bp = np.max(pos_array) / 1000  # in kb
        model_flag = "L"

        with open(filename, "w") as f:
            f.write(f"{num_snps} {chrom_range_bp} {model_flag}\n")
            for pos in pos_array:
                pos_kb = pos / 1000
                f.write(f"{pos_kb:.3f}\n")

        print(f"Saved: {filename}")

Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Samples_50/Pf3D7_01_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Samples_50/Pf3D7_02_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Samples_50/Pf3D7_03_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Samples_50/Pf3D7_04_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Samples_50/Pf3D7_05_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Samples_50/Pf3D7_06_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/LD_hat_rho_mu_data/Samples_50/Pf3D7_07_v3_seg_sites_MaeSot.txt
Saved: /data/proj2/home/students/u.srinivasan/Plasmodium_us/In